# Schema Dump

Generate a typed key tree from an EU5 save — keys, types, sizes, and sample values. Useful for documenting save structure without wading through raw JSON.

Supports full-save dumps or focused section dumps (e.g. just `countries.database.2186`).

In [ ]:
# Setup
import sys, os, json
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# === CONFIGURATION ===
SAVE_FILE   = "saves/autosave.eu5"
RAKALY_BIN  = "bin/rakaly/rakaly"

In [ ]:
from toolbox.save_loader import load_save
from toolbox.schema_dump import _schema_node, navigate_path

save = load_save(SAVE_FILE, rakaly_bin=RAKALY_BIN, verbose=True)
data = save.raw
print("Save loaded.")

## Full Top-Level Schema

In [ ]:
# Dump top-level schema (depth 2 = overview, depth 4 = detailed)
DEPTH = 2

schema = _schema_node(data, 0, DEPTH)
print(json.dumps(schema, indent=2, ensure_ascii=False)[:5000])
print("\n... (truncated for display)")

## Section-Focused Dump

Navigate to a specific section and dump its schema at higher depth.

In [ ]:
# Change SECTION and DEPTH to explore different parts
SECTION = "countries.database.2186"    # dot-separated path
DEPTH   = 4

try:
    section_data = navigate_path(data, SECTION)
    section_schema = _schema_node(section_data, 0, DEPTH)
    print(f"Schema for: {SECTION} (depth {DEPTH})\n")
    print(json.dumps(section_schema, indent=2, ensure_ascii=False)[:8000])
except KeyError as e:
    print(f"Path error: {e}")

## Export to File

In [ ]:
# Export a schema to a JSON file
OUTPUT_FILE = "docs/games/eu5/schema_dump.json"
EXPORT_DEPTH = 4
EXPORT_SECTION = None     # Set to e.g. "countries.database.2186" for a section, or None for full save

export_data = data
if EXPORT_SECTION:
    export_data = navigate_path(data, EXPORT_SECTION)

export_schema = _schema_node(export_data, 0, EXPORT_DEPTH)
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(export_schema, f, indent=2, ensure_ascii=False)

print(f"Schema written to {OUTPUT_FILE}")